# Data Workflow: Database and Market Data Integration

## Overview

This notebook explains the data layer behind the fund risk monitoring workflows.

It is not a risk report. It shows how the repository builds a controlled fund dataset, stores it in SQLite, enriches the raw position data, and makes the resulting data available to the risk analytics modules.

The project uses controlled mock data with a fixed reference date of `2026-05-13` so the examples are reproducible.

In [ ]:
from src.config import VALUATION_DATE
VALUATION_DATE





## Full Data Workflow

The workflow has four layers:

1. **Source data layer**

   * controlled JSON inputs under `reference_data/`
   * generated position files under `data/`

2. **SQLite database layer**

   * SQLite database created by `setup_db.py`
   * fund metadata and raw position data loaded into the database
   * specialised tables created for private equity and infrastructure examples

3. **Market data and enrichment layer**

   * Bloomberg-style market data interface
   * `bdp()` for point-in-time fields
   * `bdh()` for historical series
   * enrichment of raw positions with market-data and fund-admin fields

4. **Analytics input layer**

   * risk-ready DataFrame loaded from the database
   * input for VaR, stress, liquidity, leverage, and reporting workflows


### 1. Source data layer

The repository uses two types of source data, these are JSON files under `reference_data/portfolios` and act as the source-of-truth inputs for the controlled example dataset. 

For position-based funds, such as hedge fund, private debt, real estate, and UCITS examples, the holdings are defined in `position_specs/`. These specifications are used to generate reproducible position files under `data/`, which are then loaded into the SQLite `positions` table.

In [ ]:
ls ../../reference_data/portfolios/position_specs

Private equity and infrastructure examples follow a separate workflow, as they require different data models. Their data is generated by dedicated fund-generation functions and stored in specialised SQLite tables such as `pe_*` and `infra_*`.

In [ ]:
ls ../../reference_data/portfolios/*.json

## 2. SQLite database layer

The SQLite database is created by running `setup_db.py`.

This script creates the database structure, loads fund metadata, and populates the raw `positions` table from the generated position files. At this stage, the database provides the core fund and position data used by the notebooks.

The database is created locally at: `<project_root>/data/risk_management.db`

The database only needs to be recreated when the input data or database structure changes OR at first time you run this exercise.

In [ ]:
# Setup (do this once or when rebuilding the database)
from src.setup_db import run as setup_db

# This creates/rebuilds the database from reference_data/ and Excel position files
# setup_db(force=True)  # Uncomment to rebuild (takes ~10 seconds)

print(f'Valuation date: {VALUATION_DATE}')
print('Database ready at: data/risk_management.db')

### Database schema

The SQLite database contains a common position-based model and specialised models for private equity and infrastructure.

| Group | Tables | Purpose |
|---|---|---|
| Fund metadata | `funds` | Metadata for the four position-based funds: hedge fund, private debt, real estate, and UCITS |
| Position-based model | `positions`, `positions_enriched`, `instruments` | Raw and enriched holdings used by the position-based risk workflows |
| Private equity model | `pe_funds`, `pe_portfolio_companies`, `pe_fund_investments`, `pe_cash_flows`, `pe_nav_history`, `pe_valuation_report`, `pe_company_metrics`, `pe_fund_cash_management` | Private equity fund structure, portfolio companies, cash flows, valuations, and NAV history |
| Infrastructure model | `infra_funds`, `infra_assets`, `infra_fund_investments`, `infra_cash_flows`, `infra_nav_history`, `infra_valuation_report`, `infra_debt`, `infra_covenants` | Infrastructure fund structure, assets, cash flows, valuations, debt, and covenant monitoring |

The database represents six logical funds in total: four position-based funds, one private equity fund, and one infrastructure fund.

### Connecting to the database

Once the database has been created, it can be accessed through the repository's database utilities in `src.data.database`.

The repository provides a helper function, `get_engine()`, which returns a configured SQLAlchemy engine. This centralises database configuration and avoids repeating connection setup throughout the notebooks.

In [ ]:
from src.data.database import get_engine

engine = get_engine()
type(engine)

### Querying the database

Once connected, the database can be queried directly using SQLAlchemy and standard SQL. Se following examples.

In [ ]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

from sqlalchemy import text
import pandas as pd

pd.read_sql(
    text("""
    SELECT *
    FROM funds
    ORDER BY fund_id
    LIMIT 10
    """),
    engine,
)

In [ ]:
fund_id = "AIFM_HedgeFund"  # ← Full fund ID, not 'AIFM_HF'
valuation_date = "2026-05-13"

query = text("""
SELECT *
FROM positions
WHERE fund_id = :fund_id
  AND date = :date
""")

positions = pd.read_sql(
    query,
    engine,
    params={
        "fund_id": fund_id,
        "date": valuation_date,  
    },
)

positions.head(2)


In [ ]:
infra_assets = pd.read_sql(
    text("""
    SELECT a.*, fi.entry_date, fi.ownership_pct, fi.cost_basis_eur
    FROM infra_assets a
    JOIN infra_fund_investments fi ON a.asset_id = fi.asset_id
    WHERE fi.fund_id = 'AIFM_Infra_Core'
    ORDER BY a.asset_id
    LIMIT 5
    """),
    engine
)
infra_assets

Thoutgh repository remains fully accessible through SQL, most notebooks use the **reusable query functions** provided by the repository rather than embedding SQL directly in notebook cells. This keeps the notebooks focused on risk management workflows and reporting while centralising common data access logic in a single location.

Examples include helper functions for retrieving positions, NAV history, enriched risk-ready datasets, and other frequently used data structures, as will be demonstrated later in this notebook.

**Ex. 1: Querying fund positions**

The primary query function for daily position snapshots:

In [ ]:
from src.data.database import query_positions
import pandas as pd

# Get all positions for one fund on one date
fund_id = 'AIFM_HedgeFund'
valuation_date = '2026-05-13'

positions = query_positions(engine, fund_id, valuation_date)
positions.head(2)

**Ex. 2: Querying historical NAV and P&L**

In [ ]:
from src.data.database import query_nav_history
# Load 250-day NAV history
nav_hist = query_nav_history(engine, 'AIFM_HedgeFund')
nav_hist['date'] = pd.to_datetime(nav_hist['date'])

print(f'NAV history: {len(nav_hist)} days')
nav_hist.head(2)

## 3. Market data and enrichment layer

The repository includes a lightweight market data interface that mirrors the type of workflow normally handled through Bloomberg or another market data vendor.

The interface exposes Bloomberg-style methods:

- `bdp()` for point-in-time fields
- `bdh()` for historical series

It also uses Bloomberg-style field names, such as:

- `PX_LAST`
- `BETA`
- `DUR_ADJ_MID`
- `CONVEXITY`
- `YLD_YTM_MID`

In this project, the interface is backed by controlled mock data, `yfinance` where appropriate, and local caching for downloaded time series. This keeps the examples reproducible while keeping the analytics code independent from the underlying data source.

The same workflow can therefore use the mock interface in this repository and a real vendor adapter in a production setting.



In [ ]:
from src.data.mock_bloomberg import MockBloomberg

# Create a mock Bloomberg instance
bbg = MockBloomberg(seed=42)  # seed for reproducibility

# from the object below we can call methods like bbg.get_price('AAPL', '2026-05-13') to get mock price data
type(bbg)

### bdp: Bloomberg Data Point (static fields)

Get current or latest field values for a security:

In [ ]:
# equity example
fields = ['BETA', 'NAME', 'CRNCY', 'VOLUME_AVG_20D']
result_eq = bbg.bdp('AAPL US Equity', fields)
print('AAPL US Equity:')
print(result_eq)
print()

# bond example
fields_bond = ['DUR_ADJ_MID', 'CONVEXITY', 'YLD_YTM_MID']
result_bond = bbg.bdp('US912828YK09 Govt', fields_bond)  
print('Bond fields:')
print(result_bond)

# examples to test different security types and field groups
# DBR 0 08/15/29 Govt — Bund
# XS2543791470 Corp — Corporate bond

### bdh: Bloomberg Data History (time series)

Get historical prices for backtesting:

In [ ]:
# Get 60 days of price history
hist = bbg.bdh('AAPL US Equity', 'PX_LAST', '2026-03-01', '2026-05-13')
print(f'AAPL price history: {len(hist)} days')
print('...')
print(hist.tail(5))

<small>
Note: Historical market data downloaded through the mock market data interface may be cached locally under <code>data/yf_cache/</code>. The cache is only a development convenience and can be refreshed when the requested date range is not covered.
</small>


### Enrichment

Raw positions from the database have quantity and price, but lack risk sensitivities (beta, duration, etc.). Enrichment adds these sensitivities from two sources:

**For liquid assets** (equities, bonds, FX with Bloomberg ticker):
- Call MockBloomberg to fetch sensitivities (BETA, DUR_ADJ_MID, CONVEXITY, etc.)
- Store in `positions_enriched` table

    | Asset Class | Columns | Source |
    |-------------|---------|--------|
    | Equity | beta, eqy_dvd_yld_ind, volume_avg_20d | Bloomberg |
    | Bond | dur_adj_mid, convexity, yld_ytm_mid, rtg_sp | Bloomberg |
    | FX | px_last (FX rate) | Bloomberg |
    | Direct Property | ltv_pct, rental_yield_pct, vacancy_rate_pct | Fund admin (already in positions) |
    | Loan | credit spread, maturity | Fund admin |
    | Derivative | delta, convexity, underlying price | Bloomberg (if applicable) |


**For illiquid assets** (direct property, private loans with no Bloomberg ticker):
- Use fund-admin data already in the positions table (ltv_pct, rental_yield_pct, etc.)
- Mark source as 'fund_admin'


In [ ]:
position_based = pd.read_sql(
    text("""
    SELECT fund_id FROM funds 
    WHERE fund_id NOT IN ('AIFM_PE_Buyout', 'AIFM_Infra_Core')
    ORDER BY fund_id
    """),
    engine
)
position_based

In [ ]:
from src.data.enrichment import enrich_positions

# This calls MockBloomberg for each position with a ticker
enrich_positions(engine, fund_id=fund_id, date='2026-05-13')  # Uncomment to run (takes ~5 seconds per fund)

## 4. Analytics input layer

`get_risk_ready_df()` loads the enriched position data into a pandas DataFrame.

```
Computation modules (src/computation/):
  ├─ var.py:          var_historical(positions_df, ...)
  ├─ stress.py:       stress_equity(positions_df, ...)
  ├─ liquidity.py:    liquidity_buckets(positions_df, ...)
  ├─ leverage.py:     compute_leverage(positions_df, ...)
  └─ attribution.py:  compute_pnl_attribution(positions_df, ...)
      ↓
Risk metrics (VaR, ES, stress outcomes, liquidity, leverage)
      ↓
Reporting (board_report, annex_iv) / Notebooks
```


In [ ]:
from src.data.enrichment import get_risk_ready_df

# Get enriched positions for a fund on a date
# This joins raw positions with enriched sensitivities
risk_df = get_risk_ready_df(engine, 'AIFM_HedgeFund', '2026-05-13')

print(f'Enriched positions: {len(risk_df)} rows')
print()
print('Key columns for risk computation:')
key_cols = ['isin', 'instrument_name', 'asset_class', 'market_value_eur', 'beta', 'dur_adj_mid', 'currency']
risk_df[key_cols].head()


#